# Div Breakout RR — Backtrader Research Notebook

Чистовая структура ноутбука для проверки PineScript-логики в Python, backtest, OOS, Optuna и WFA.

## Навигация

1. Environment & config
2. Data loading
3. Indicators
4. Backtrader feed
5. Strategy implementation
6. Backtest runner & metrics
7. Out-of-sample split
8. Optuna optimization
9. Walk-forward analysis
10. WFA vs fixed baseline
11. Rolling validation candidates
12. Risk sweep
13. Commission stress-test

> Торговая логика стратегии не менялась. Ноутбук очищен от выводов и разложен по разделам.


## 1. Environment & config

In [ ]:
!pip install backtrader optuna pandas numpy matplotlib

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict, replace
from typing import Optional, Dict, Any, List, Tuple

import warnings

import numpy as np
import pandas as pd

try:
    import backtrader as bt
except ImportError as exc:
    raise ImportError("Установи Backtrader: pip install backtrader") from exc

try:
    import optuna
except ImportError:
    optuna = None
    warnings.warn("Optuna не установлена. Оптимизационные ячейки будут работать после: pip install optuna")

In [ ]:
@dataclass(frozen=True)
class StrategyParams:
    """Параметры, соответствующие PineScript inputs."""
    rsi_len: int = 13
    left_bars: int = 3
    right_bars: int = 5
    rr: float = 1.5
    risk_pct: float = 3.0
    stop_buffer_pct: float = 0.0
    stop_mode: str = "Nearest pivot"  # "Nearest pivot" or "Deepest pivot"
    max_setup_bars: int = 90

    # Не параметр PineScript, а настройка совместимости pivot ties.
    # False: pivot = max/min окна, совпадения допускаются. Для BTC обычно неважно.
    strict_pivots: bool = False

    def validate(self) -> None:
        if self.rsi_len < 2:
            raise ValueError("rsi_len должен быть >= 2")
        if self.left_bars < 1 or self.right_bars < 1:
            raise ValueError("left_bars/right_bars должны быть >= 1")
        if self.rr <= 0:
            raise ValueError("rr должен быть > 0")
        if self.risk_pct <= 0:
            raise ValueError("risk_pct должен быть > 0")
        if self.stop_buffer_pct < 0:
            raise ValueError("stop_buffer_pct должен быть >= 0")
        if self.stop_mode not in {"Nearest pivot", "Deepest pivot"}:
            raise ValueError("stop_mode должен быть 'Nearest pivot' или 'Deepest pivot'")


BASELINE_PARAMS = StrategyParams(
    rsi_len=13,
    left_bars=3,
    right_bars=5,
    rr=1.5,
    risk_pct=3.0,
    stop_buffer_pct=0.0,
    stop_mode="Nearest pivot",
    max_setup_bars=90,
)

BASELINE_PARAMS

WFA_BASE_PARAMS = StrategyParams(
    rsi_len=13,
    left_bars=3,
    right_bars=5,
    rr=1.5,
    risk_pct=3.0,
    stop_buffer_pct=0.0,
    stop_mode="Nearest pivot",
    max_setup_bars=90,
    strict_pivots=False,
)
WFA_BASE_PARAMS

## 2. Data loading

In [ ]:
import requests
import pandas as pd
import time


def _to_utc_ms(value):
    ts = pd.Timestamp(value)

    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")

    return int(ts.timestamp() * 1000)


def load_bybit_klines(
    category="spot",
    symbol="BTCUSDT",
    interval="15",
    start_time="2026-03-01 00:00:00",
    end_time="2026-05-11 00:00:00",
    pause=0.2,
    max_pages=1000,
):
    """
    Загружает OHLCV с Bybit.

    Важно:
    Bybit возвращает свечи от новых к старым, поэтому пагинация идёт назад через current_end.
    """

    url = "https://api.bybit.com/v5/market/kline"

    start_ts = _to_utc_ms(start_time)
    end_ts = _to_utc_ms(end_time)

    if end_ts <= start_ts:
        raise ValueError("end_time должен быть позже start_time")

    all_rows = []
    current_end = end_ts

    session = requests.Session()

    for page in range(max_pages):
        params = {
            "category": category,
            "symbol": symbol,
            "interval": str(interval),
            "start": start_ts,
            "end": current_end,
            "limit": 1000,
        }

        response = session.get(url, params=params, timeout=30)
        data = response.json()

        if data.get("retCode") != 0:
            raise RuntimeError(f"Bybit API error: {data}")

        rows = data.get("result", {}).get("list", [])

        if not rows:
            break

        rows_sorted = sorted(rows, key=lambda x: int(x[0]))
        all_rows.extend(rows_sorted)

        oldest_start_time = int(rows_sorted[0][0])

        if oldest_start_time <= start_ts:
            break

        new_current_end = oldest_start_time - 1

        if new_current_end >= current_end:
            raise RuntimeError(
                f"Pagination stuck: current_end={current_end}, "
                f"new_current_end={new_current_end}"
            )

        current_end = new_current_end
        time.sleep(pause)

    else:
        raise RuntimeError(f"Reached max_pages={max_pages}. Possible pagination issue.")

    df = pd.DataFrame(
        all_rows,
        columns=["open_time", "open", "high", "low", "close", "volume", "turnover"]
    )

    if df.empty:
        return df

    for col in ["open", "high", "low", "close", "volume", "turnover"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["open_time"] = df["open_time"].astype("int64")

    df["datetime"] = (
        pd.to_datetime(df["open_time"], unit="ms", utc=True)
        .dt.tz_convert(None)
    )

    df = (
        df.sort_values("datetime")
        .drop_duplicates(subset=["open_time"])
        .reset_index(drop=True)
    )

    start_dt = pd.Timestamp(start_time)
    end_dt = pd.Timestamp(end_time)

    if start_dt.tzinfo is not None:
        start_dt = start_dt.tz_convert(None)
    if end_dt.tzinfo is not None:
        end_dt = end_dt.tz_convert(None)

    df = df[(df["datetime"] >= start_dt) & (df["datetime"] <= end_dt)]
    df = df.reset_index(drop=True)

    return df


def normalize_ohlcv_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Приводит OHLCV к формату, который ожидает Backtrader и остальные ячейки.

    Результат:
        index: DatetimeIndex
        columns: open, high, low, close, volume
    """

    df = df.copy()

    rename_map = {c: c.lower() for c in df.columns}
    df = df.rename(columns=rename_map)

    required = ["open", "high", "low", "close"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Нет обязательных колонок: {missing}")

    if "volume" not in df.columns:
        df["volume"] = 0.0

    if not isinstance(df.index, pd.DatetimeIndex):
        datetime_candidates = ["datetime", "date", "time", "timestamp"]
        found = next((c for c in datetime_candidates if c in df.columns), None)

        if found is None:
            raise ValueError("Нужен DatetimeIndex или колонка datetime/date/time/timestamp")

        df[found] = pd.to_datetime(df[found], utc=True, errors="coerce")
        df = df.set_index(found)

    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")]

    # Backtrader обычно стабильнее работает с timezone-naive DatetimeIndex.
    if df.index.tz is not None:
        df.index = df.index.tz_convert(None)

    numeric_cols = ["open", "high", "low", "close", "volume"]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["open", "high", "low", "close"])

    return df[numeric_cols]


def parse_ohlcv_data() -> pd.DataFrame:
    """
    Главная функция загрузки данных для ноутбука.

    Меняешь здесь:
        category
        symbol
        interval
        start_time
        end_time

    Возвращает уже нормализованный df:
        df.index -> DatetimeIndex
        df.columns -> open, high, low, close, volume
    """

    raw = load_bybit_klines(
        category="spot",
        symbol="BTCUSDT",
        interval="15",
        start_time="2024-01-01 00:00:00",
        end_time="2026-05-11 00:00:00",
        pause=0.2,
    )

    df = normalize_ohlcv_df(raw)

    return df

In [ ]:
# Загрузка данных
df = parse_ohlcv_data()

df.head(), df.tail(), df.shape


## 3. Indicators

In [ ]:
def tv_rma(series: pd.Series, length: int) -> pd.Series:
    """
    PineScript ta.rma equivalent:
    - первое значение = SMA по первым length ненулевым наблюдениям;
    - дальше: rma = alpha * x + (1 - alpha) * prev, alpha = 1 / length.
    """
    if length <= 0:
        raise ValueError("length должен быть > 0")

    x = pd.Series(series, dtype="float64").copy()
    out = pd.Series(np.nan, index=x.index, dtype="float64")
    alpha = 1.0 / length

    valid = x.dropna()
    if len(valid) < length:
        return out

    first_idx = valid.index[length - 1]
    first_value = valid.iloc[:length].mean()
    out.loc[first_idx] = first_value

    prev = first_value
    started = False
    for idx, value in x.items():
        if idx == first_idx:
            started = True
            continue
        if not started:
            continue
        if np.isnan(value):
            out.loc[idx] = prev
            continue
        prev = alpha * value + (1.0 - alpha) * prev
        out.loc[idx] = prev

    return out


def tv_rsi(close: pd.Series, length: int) -> pd.Series:
    """PineScript ta.rsi(close, length) approximation with Wilder RMA."""
    close = pd.Series(close, dtype="float64")
    change = close.diff()
    up = change.clip(lower=0)
    down = (-change).clip(lower=0)

    rma_up = tv_rma(up, length)
    rma_down = tv_rma(down, length)

    rs = rma_up / rma_down
    rsi = 100.0 - (100.0 / (1.0 + rs))

    # TradingView-like edge cases.
    rsi = rsi.where(rma_down != 0, 100.0)
    rsi = rsi.where(rma_up != 0, 0.0)
    rsi = rsi.where(~((rma_up == 0) & (rma_down == 0)), 50.0)

    return rsi


def prepare_backtrader_df(df: pd.DataFrame, params: StrategyParams) -> pd.DataFrame:
    """Добавляет tv_rsi и чистит данные перед Backtrader."""
    params.validate()
    out = normalize_ohlcv_df(df)
    out["tv_rsi"] = tv_rsi(out["close"], params.rsi_len)
    out = out.dropna(subset=["tv_rsi"])
    return out

## 4. Backtrader feed

In [ ]:
class PandasTVData(bt.feeds.PandasData):
    lines = ("tv_rsi",)
    params = (
        ("datetime", None),
        ("open", "open"),
        ("high", "high"),
        ("low", "low"),
        ("close", "close"),
        ("volume", "volume"),
        ("openinterest", None),
        ("tv_rsi", "tv_rsi"),
    )

## 5. Strategy implementation

In [ ]:

class DivBreakoutRiskStrategy(bt.Strategy):
    params = dict(
        rsi_len=13,
        left_bars=3,
        right_bars=5,
        rr=1.5,
        risk_pct=3.0,
        stop_buffer_pct=0.0,
        stop_mode="Nearest pivot",
        max_setup_bars=90,
        strict_pivots=False,
        printlog=False,
    )

    def __init__(self):
        self.prev_price_low = None
        self.prev_rsi_low = None
        self.prev_low_high = None
        self.prev_low_index = None

        self.prev_price_high = None
        self.prev_rsi_high = None
        self.prev_high_low = None
        self.prev_high_index = None

        self.bull_active = False
        self.bull_trigger = None
        self.bull_stop = None
        self.bull_take = None
        self.bull_setup_bar = None

        self.bear_active = False
        self.bear_trigger = None
        self.bear_stop = None
        self.bear_take = None
        self.bear_setup_bar = None

        self.entry_order = None
        self.stop_order = None
        self.limit_order = None
        self.entry_meta: Dict[int, Dict[str, Any]] = {}

        self.equity_curve: List[Dict[str, Any]] = []
        self.trade_records: List[Dict[str, Any]] = []
        self.event_log: List[Dict[str, Any]] = []
        self.current_trade_side: Optional[str] = None
        self.current_trade_entry_dt = None
        self.current_trade_entry_bar = None
        self.current_trade_entry_price = None

    def log(self, txt: str) -> None:
        if self.p.printlog:
            dt = self.data.datetime.datetime(0)
            print(f"{dt.isoformat()} {txt}")

    @property
    def bar_index(self) -> int:
        return len(self.data) - 1

    def _enough_for_pivot(self) -> bool:
        return len(self.data) >= (self.p.left_bars + self.p.right_bars + 1)

    def _is_pivot_low_now(self) -> bool:
        rb = self.p.right_bars
        lb = self.p.left_bars
        pivot = float(self.data.low[-rb])
        for off in range(-rb - lb, 1):
            if off == -rb:
                continue
            value = float(self.data.low[off])
            if self.p.strict_pivots:
                if value <= pivot:
                    return False
            else:
                if value < pivot:
                    return False
        return True

    def _is_pivot_high_now(self) -> bool:
        rb = self.p.right_bars
        lb = self.p.left_bars
        pivot = float(self.data.high[-rb])
        for off in range(-rb - lb, 1):
            if off == -rb:
                continue
            value = float(self.data.high[off])
            if self.p.strict_pivots:
                if value >= pivot:
                    return False
            else:
                if value > pivot:
                    return False
        return True

    def _qty_from_risk(self, entry: float, stop: float) -> float:
        risk_per_unit = abs(entry - stop)
        if risk_per_unit <= 0:
            return float("nan")
        equity = max(float(self.broker.getvalue()), 0.0)
        risk_cash = equity * float(self.p.risk_pct) / 100.0
        return risk_cash / risk_per_unit

    def _submit_entry(self, side: str, qty: float, stop: float, take: float) -> None:
        if side == "long":
            order = self.buy(size=qty)
        elif side == "short":
            order = self.sell(size=qty)
        else:
            raise ValueError(side)

        self.entry_order = order
        self.entry_meta[order.ref] = dict(side=side, stop=stop, take=take, qty=qty)

    def _detect_bullish_divergence(self) -> None:
        rb = self.p.right_bars
        curr_index = self.bar_index - rb
        curr_price_low = float(self.data.low[-rb])
        curr_rsi_low = float(self.data.tv_rsi[-rb])
        curr_high = float(self.data.high[-rb])

        bullish_div = (
            self.prev_price_low is not None
            and curr_price_low < self.prev_price_low
            and curr_rsi_low > self.prev_rsi_low
        )

        if bullish_div:
            self.bull_active = True
            self.bull_trigger = self.prev_low_high
            self.bull_setup_bar = self.bar_index

            if self.p.stop_mode == "Nearest pivot":
                self.bull_stop = curr_price_low * (1.0 - self.p.stop_buffer_pct / 100.0)
            else:
                self.bull_stop = min(self.prev_price_low, curr_price_low) * (1.0 - self.p.stop_buffer_pct / 100.0)

            self.event_log.append(dict(
                dt=self.data.datetime.datetime(0),
                event="bullish_divergence",
                pivot_dt=self.data.datetime.datetime(-rb),
                trigger=self.bull_trigger,
                stop=self.bull_stop,
                price=curr_price_low,
                rsi=curr_rsi_low,
            ))

        self.prev_price_low = curr_price_low
        self.prev_rsi_low = curr_rsi_low
        self.prev_low_high = curr_high
        self.prev_low_index = curr_index

    def _detect_bearish_divergence(self) -> None:
        rb = self.p.right_bars
        curr_index = self.bar_index - rb
        curr_price_high = float(self.data.high[-rb])
        curr_rsi_high = float(self.data.tv_rsi[-rb])
        curr_low = float(self.data.low[-rb])

        bearish_div = (
            self.prev_price_high is not None
            and curr_price_high > self.prev_price_high
            and curr_rsi_high < self.prev_rsi_high
        )

        if bearish_div:
            self.bear_active = True
            self.bear_trigger = self.prev_high_low
            self.bear_setup_bar = self.bar_index

            if self.p.stop_mode == "Nearest pivot":
                self.bear_stop = curr_price_high * (1.0 + self.p.stop_buffer_pct / 100.0)
            else:
                self.bear_stop = max(self.prev_price_high, curr_price_high) * (1.0 + self.p.stop_buffer_pct / 100.0)

            self.event_log.append(dict(
                dt=self.data.datetime.datetime(0),
                event="bearish_divergence",
                pivot_dt=self.data.datetime.datetime(-rb),
                trigger=self.bear_trigger,
                stop=self.bear_stop,
                price=curr_price_high,
                rsi=curr_rsi_high,
            ))

        self.prev_price_high = curr_price_high
        self.prev_rsi_high = curr_rsi_high
        self.prev_high_low = curr_low
        self.prev_high_index = curr_index

    def _invalidate_setups(self) -> None:
        if self.bull_active:
            too_old = (self.bar_index - self.bull_setup_bar) > self.p.max_setup_bars
            stop_broken = float(self.data.low[0]) <= self.bull_stop
            if too_old or stop_broken:
                self.bull_active = False
                self.event_log.append(dict(
                    dt=self.data.datetime.datetime(0),
                    event="bull_setup_cancelled",
                    reason="too_old" if too_old else "stop_broken",
                ))

        if self.bear_active:
            too_old = (self.bar_index - self.bear_setup_bar) > self.p.max_setup_bars
            stop_broken = float(self.data.high[0]) >= self.bear_stop
            if too_old or stop_broken:
                self.bear_active = False
                self.event_log.append(dict(
                    dt=self.data.datetime.datetime(0),
                    event="bear_setup_cancelled",
                    reason="too_old" if too_old else "stop_broken",
                ))

    def _check_entries(self) -> None:
        if self.position or self.entry_order is not None:
            return

        close = float(self.data.close[0])
        long_signal = self.bull_active and close > self.bull_trigger
        short_signal = self.bear_active and close < self.bear_trigger

        if long_signal:
            entry_price = close
            risk = entry_price - self.bull_stop
            if risk > 0:
                qty = self._qty_from_risk(entry_price, self.bull_stop)
                if np.isfinite(qty) and qty > 0:
                    take = entry_price + risk * self.p.rr
                    self._submit_entry("long", qty, self.bull_stop, take)
                    self.event_log.append(dict(
                        dt=self.data.datetime.datetime(0),
                        event="long_signal",
                        entry_estimate=entry_price,
                        stop=self.bull_stop,
                        take=take,
                        qty=qty,
                    ))
                self.bull_active = False
            return

        if short_signal:
            entry_price = close
            risk = self.bear_stop - entry_price
            if risk > 0:
                qty = self._qty_from_risk(entry_price, self.bear_stop)
                if np.isfinite(qty) and qty > 0:
                    take = entry_price - risk * self.p.rr
                    self._submit_entry("short", qty, self.bear_stop, take)
                    self.event_log.append(dict(
                        dt=self.data.datetime.datetime(0),
                        event="short_signal",
                        entry_estimate=entry_price,
                        stop=self.bear_stop,
                        take=take,
                        qty=qty,
                    ))
                self.bear_active = False

    def next(self):
        self.equity_curve.append(dict(
            dt=self.data.datetime.datetime(0),
            equity=float(self.broker.getvalue()),
            cash=float(self.broker.getcash()),
            close=float(self.data.close[0]),
        ))

        if not self._enough_for_pivot():
            return

        if np.isnan(float(self.data.tv_rsi[-self.p.right_bars])):
            return

        if self._is_pivot_low_now():
            self._detect_bullish_divergence()

        if self._is_pivot_high_now():
            self._detect_bearish_divergence()

        self._invalidate_setups()
        self._check_entries()

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return

        if order.ref in self.entry_meta:
            meta = self.entry_meta.pop(order.ref)
            self.entry_order = None

            if order.status == order.Completed:
                side = meta["side"]
                size = abs(float(order.executed.size))
                stop = float(meta["stop"])
                take = float(meta["take"])

                self.current_trade_side = side
                self.current_trade_entry_dt = self.data.datetime.datetime(0)
                self.current_trade_entry_bar = self.bar_index
                self.current_trade_entry_price = float(order.executed.price)

                if side == "long":
                    self.stop_order = self.sell(size=size, exectype=bt.Order.Stop, price=stop)
                    self.limit_order = self.sell(size=size, exectype=bt.Order.Limit, price=take, oco=self.stop_order)
                else:
                    self.stop_order = self.buy(size=size, exectype=bt.Order.Stop, price=stop)
                    self.limit_order = self.buy(size=size, exectype=bt.Order.Limit, price=take, oco=self.stop_order)

                self.event_log.append(dict(
                    dt=self.data.datetime.datetime(0),
                    event=f"{side}_entry_filled",
                    fill_price=float(order.executed.price),
                    size=size,
                    stop=stop,
                    take=take,
                ))
            else:
                self.event_log.append(dict(
                    dt=self.data.datetime.datetime(0),
                    event="entry_rejected_or_cancelled",
                    status=order.getstatusname(),
                    meta=meta,
                ))

        if order.status in [order.Completed, order.Canceled, order.Margin, order.Rejected]:
            if order == self.stop_order:
                self.stop_order = None
            if order == self.limit_order:
                self.limit_order = None

    def notify_trade(self, trade):
        if trade.isclosed:
            self.trade_records.append(dict(
                entry_dt=self.current_trade_entry_dt,
                exit_dt=self.data.datetime.datetime(0),
                side=self.current_trade_side,
                entry_price=self.current_trade_entry_price,
                pnl=float(trade.pnl),
                pnlcomm=float(trade.pnlcomm),
                bars=(self.bar_index - self.current_trade_entry_bar) if self.current_trade_entry_bar is not None else None,
            ))
            self.current_trade_side = None
            self.current_trade_entry_dt = None
            self.current_trade_entry_bar = None
            self.current_trade_entry_price = None


## 6. Backtest runner & metrics

In [ ]:
def _max_drawdown_pct(equity: pd.Series) -> float:
    if equity.empty:
        return float("nan")
    roll_max = equity.cummax()
    dd = equity / roll_max - 1.0
    return abs(float(dd.min())) * 100.0


def _trade_metrics(trades: pd.DataFrame, prefix: str = "") -> Dict[str, Any]:
    if trades.empty:
        return {
            f"{prefix}total_trades": 0,
            f"{prefix}winning_trades": 0,
            f"{prefix}losing_trades": 0,
            f"{prefix}winrate_pct": 0.0,
            f"{prefix}gross_profit": 0.0,
            f"{prefix}gross_loss": 0.0,
            f"{prefix}profit_factor": 0.0,
            f"{prefix}avg_pnl": 0.0,
            f"{prefix}avg_win": 0.0,
            f"{prefix}avg_loss": 0.0,
        }

    pnl = trades["pnlcomm"].astype(float)
    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]
    gross_profit = float(wins.sum())
    gross_loss = float(-losses.sum())
    pf = float("inf") if gross_loss == 0 and gross_profit > 0 else (gross_profit / gross_loss if gross_loss > 0 else 0.0)

    return {
        f"{prefix}total_trades": int(len(trades)),
        f"{prefix}winning_trades": int((pnl > 0).sum()),
        f"{prefix}losing_trades": int((pnl < 0).sum()),
        f"{prefix}winrate_pct": float((pnl > 0).mean() * 100.0),
        f"{prefix}gross_profit": gross_profit,
        f"{prefix}gross_loss": gross_loss,
        f"{prefix}profit_factor": pf,
        f"{prefix}avg_pnl": float(pnl.mean()),
        f"{prefix}avg_win": float(wins.mean()) if len(wins) else 0.0,
        f"{prefix}avg_loss": float(losses.mean()) if len(losses) else 0.0,
    }


def compute_metrics(
    equity_curve: List[Dict[str, Any]],
    trade_records: List[Dict[str, Any]],
    start_cash: float,
) -> Dict[str, Any]:
    eq = pd.DataFrame(equity_curve)
    trades = pd.DataFrame(trade_records)

    if eq.empty:
        final_equity = start_cash
        max_dd = float("nan")
    else:
        eq = eq.set_index("dt")
        final_equity = float(eq["equity"].iloc[-1])
        max_dd = _max_drawdown_pct(eq["equity"])

    metrics: Dict[str, Any] = {
        "start_cash": float(start_cash),
        "final_equity": final_equity,
        "net_pnl": final_equity - float(start_cash),
        "net_pnl_pct": (final_equity / float(start_cash) - 1.0) * 100.0,
        "max_drawdown_pct": max_dd,
    }

    metrics.update(_trade_metrics(trades, prefix=""))

    if not trades.empty and "side" in trades.columns:
        metrics.update(_trade_metrics(trades[trades["side"] == "long"], prefix="long_"))
        metrics.update(_trade_metrics(trades[trades["side"] == "short"], prefix="short_"))
    else:
        metrics.update(_trade_metrics(pd.DataFrame(), prefix="long_"))
        metrics.update(_trade_metrics(pd.DataFrame(), prefix="short_"))

    return metrics


def run_backtest(
    df: pd.DataFrame,
    params: StrategyParams = BASELINE_PARAMS,
    start_cash: float = 10_000.0,
    commission: float = 0.0004,  # 0.04% like Pine commission_value=0.04
    broker_leverage: float = 100.0,
    cheat_on_close: bool = True,
    printlog: bool = False,
) -> Tuple[Dict[str, Any], DivBreakoutRiskStrategy, bt.Cerebro]:
    """
    Main runner.

    commission=0.0004 соответствует 0.04%.
    broker_leverage высокий, чтобы Backtrader не отклонял сделки из-за cash-limit.
    Для production-теста можно поставить реалистичное плечо.
    """
    params.validate()
    bt_df = prepare_backtrader_df(df, params)

    cerebro = bt.Cerebro(stdstats=False)
    cerebro.broker.setcash(start_cash)
    cerebro.broker.setcommission(commission=commission, leverage=broker_leverage)
    cerebro.broker.set_coc(cheat_on_close)

    data = PandasTVData(dataname=bt_df)
    cerebro.adddata(data)

    kwargs = asdict(params)
    kwargs["printlog"] = printlog
    cerebro.addstrategy(DivBreakoutRiskStrategy, **kwargs)

    results = cerebro.run(tradehistory=False)
    strat = results[0]
    metrics = compute_metrics(strat.equity_curve, strat.trade_records, start_cash=start_cash)
    return metrics, strat, cerebro


def metrics_to_frame(metrics_list: List[Dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(metrics_list)

In [ ]:
# Базовый запуск стратегии
metrics, strat, cerebro = run_backtest(
    df,
    BASELINE_PARAMS,
    start_cash=10_000,
    broker_leverage=100,
)

trades_df = pd.DataFrame(strat.trade_records)
events_df = pd.DataFrame(strat.event_log)

pd.Series(metrics).sort_index()


In [ ]:
pd.Series(metrics)[[
    "start_cash",
    "final_equity",
    "net_pnl",
    "net_pnl_pct",
    "max_drawdown_pct",
    "total_trades",
    "winrate_pct",
    "profit_factor",
]]

## 7. Out-of-sample split

In [ ]:
def split_train_test_by_ratio(df: pd.DataFrame, train_ratio: float = 0.7) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not 0 < train_ratio < 1:
        raise ValueError("train_ratio должен быть между 0 и 1")
    df = normalize_ohlcv_df(df)
    split_idx = int(len(df) * train_ratio)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


def split_train_test_by_date(
    df: pd.DataFrame,
    split_date: str | pd.Timestamp,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = normalize_ohlcv_df(df)
    split_ts = pd.Timestamp(split_date)
    if df.index.tz is not None and split_ts.tzinfo is None:
        split_ts = split_ts.tz_localize(df.index.tz)
    train = df.loc[df.index < split_ts].copy()
    test = df.loc[df.index >= split_ts].copy()
    return train, test


# Пример:
# train_df, test_df = split_train_test_by_ratio(df, 0.7)
# train_metrics, _, _ = run_backtest(train_df, BASELINE_PARAMS)
# test_metrics, _, _ = run_backtest(test_df, BASELINE_PARAMS)
# pd.DataFrame([train_metrics, test_metrics], index=["train", "test"])[[
#     "net_pnl_pct", "max_drawdown_pct", "total_trades", "winrate_pct", "profit_factor"
# ]]

## 8. Optuna optimization helpers

In [ ]:
def trial_to_params(
    trial: "optuna.Trial",
    base: StrategyParams = BASELINE_PARAMS,
) -> StrategyParams:
    """Диапазон оптимизации вокруг текущего рабочего ядра. Риск не оптимизируется."""
    return StrategyParams(
        rsi_len=trial.suggest_int("rsi_len", 11, 15),
        left_bars=trial.suggest_int("left_bars", 2, 4),
        right_bars=trial.suggest_int("right_bars", 4, 6),
        rr=trial.suggest_categorical("rr", [1.25, 1.5, 1.75, 2.0]),
        risk_pct=base.risk_pct,
        stop_buffer_pct=trial.suggest_categorical("stop_buffer_pct", [0.0, 0.05]),
        stop_mode=base.stop_mode,
        max_setup_bars=trial.suggest_int("max_setup_bars", 70, 110, step=10),
        strict_pivots=base.strict_pivots,
    )


def make_objective(
    train_df: pd.DataFrame,
    base: StrategyParams = BASELINE_PARAMS,
    start_cash: float = 10_000.0,
    commission: float = 0.0004,
    broker_leverage: float = 100.0,
    min_trades: int = 50,
    max_dd_pct: float = 30.0,
):
    """Objective для Optuna: PF + мягкий бонус за PnL, со штрафами за мало сделок/минус/DD."""
    if optuna is None:
        raise ImportError("Optuna не установлена: pip install optuna")

    def objective(trial: "optuna.Trial") -> float:
        params = trial_to_params(trial, base=base)

        metrics, _, _ = run_backtest(
            train_df,
            params=params,
            start_cash=start_cash,
            commission=commission,
            broker_leverage=broker_leverage,
        )

        pf = metrics.get("profit_factor", 0.0)
        net_pnl_pct = metrics.get("net_pnl_pct", 0.0)
        dd_pct = metrics.get("max_drawdown_pct", 0.0)
        trades = metrics.get("total_trades", 0)

        if not np.isfinite(pf):
            pf = 10.0

        score = float(pf)

        if np.isfinite(net_pnl_pct):
            score += net_pnl_pct * 0.01

        if trades < min_trades:
            score *= max(trades, 1) / min_trades

        if net_pnl_pct <= 0:
            score *= 0.1

        if np.isfinite(dd_pct) and dd_pct > max_dd_pct:
            score *= max_dd_pct / dd_pct

        for k, v in metrics.items():
            if isinstance(v, (int, float, np.integer, np.floating)) and np.isfinite(v):
                trial.set_user_attr(k, float(v))

        return float(score)

    return objective


def optimize_params(
    train_df: pd.DataFrame,
    base: StrategyParams = BASELINE_PARAMS,
    n_trials: int = 300,
    sampler_seed: int = 42,
    direction: str = "maximize",
) -> "optuna.Study":
    if optuna is None:
        raise ImportError("Optuna не установлена: pip install optuna")

    sampler = optuna.samplers.TPESampler(seed=sampler_seed)
    study = optuna.create_study(direction=direction, sampler=sampler)
    study.optimize(make_objective(train_df, base=base), n_trials=n_trials, show_progress_bar=True)
    return study


def study_results_frame(study: "optuna.Study") -> pd.DataFrame:
    rows = []
    for t in study.trials:
        row = dict(number=t.number, value=t.value, state=str(t.state))
        row.update(t.params)
        row.update(t.user_attrs)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("value", ascending=False)


def params_from_trial_params(
    trial_params: Dict[str, Any],
    base: StrategyParams = BASELINE_PARAMS,
) -> StrategyParams:
    """Конвертирует best_trial.params обратно в StrategyParams."""
    return StrategyParams(
        rsi_len=int(trial_params.get("rsi_len", base.rsi_len)),
        left_bars=int(trial_params.get("left_bars", base.left_bars)),
        right_bars=int(trial_params.get("right_bars", base.right_bars)),
        rr=float(trial_params.get("rr", base.rr)),
        risk_pct=float(base.risk_pct),
        stop_buffer_pct=float(trial_params.get("stop_buffer_pct", base.stop_buffer_pct)),
        stop_mode=base.stop_mode,
        max_setup_bars=int(trial_params.get("max_setup_bars", base.max_setup_bars)),
        strict_pivots=base.strict_pivots,
    )


def evaluate_train_test(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    params: StrategyParams,
    start_cash: float = 10_000.0,
    commission: float = 0.0004,
    broker_leverage: float = 100.0,
) -> pd.DataFrame:
    train_metrics, _, _ = run_backtest(train_df, params, start_cash, commission, broker_leverage)
    test_metrics, _, _ = run_backtest(test_df, params, start_cash, commission, broker_leverage)
    return pd.DataFrame([train_metrics, test_metrics], index=["train", "test"])


if optuna is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)


## 9. Walk-forward folds

In [ ]:
def make_wfa_folds_by_bars(
    df: pd.DataFrame,
    train_bars: int,
    test_bars: int,
    step_bars: Optional[int] = None,
    expanding: bool = False,
) -> List[Dict[str, Any]]:
    df = normalize_ohlcv_df(df)
    step_bars = step_bars or test_bars
    folds = []
    start = 0

    while True:
        train_start = 0 if expanding else start
        train_end = start + train_bars
        test_start = train_end
        test_end = test_start + test_bars

        if test_end > len(df):
            break

        folds.append(dict(
            train_start=df.index[train_start],
            train_end=df.index[train_end - 1],
            test_start=df.index[test_start],
            test_end=df.index[test_end - 1],
            train_df=df.iloc[train_start:train_end].copy(),
            test_df=df.iloc[test_start:test_end].copy(),
        ))

        start += step_bars

    return folds


def make_wfa_folds_by_time(
    df: pd.DataFrame,
    train_window: str | pd.Timedelta,
    test_window: str | pd.Timedelta,
    step_window: Optional[str | pd.Timedelta] = None,
    expanding: bool = False,
) -> List[Dict[str, Any]]:
    df = normalize_ohlcv_df(df)
    train_delta = pd.Timedelta(train_window)
    test_delta = pd.Timedelta(test_window)
    step_delta = pd.Timedelta(step_window) if step_window is not None else test_delta

    folds = []
    first = df.index.min()
    last = df.index.max()
    anchor = first

    while True:
        train_start = first if expanding else anchor
        train_end = anchor + train_delta
        test_start = train_end
        test_end = test_start + test_delta

        if test_end > last:
            break

        train_df = df[(df.index >= train_start) & (df.index < train_end)].copy()
        test_df = df[(df.index >= test_start) & (df.index < test_end)].copy()

        if len(train_df) > 0 and len(test_df) > 0:
            folds.append(dict(
                train_start=train_df.index.min(),
                train_end=train_df.index.max(),
                test_start=test_df.index.min(),
                test_end=test_df.index.max(),
                train_df=train_df,
                test_df=test_df,
            ))

        anchor = anchor + step_delta

    return folds


In [ ]:
folds = make_wfa_folds_by_time(
    df,
    train_window="180D",
    test_window="60D",
    step_window="60D",
    expanding=False,
)

folds_info = pd.DataFrame([
    {
        "fold": i,
        "train_start": f["train_start"],
        "train_end": f["train_end"],
        "test_start": f["test_start"],
        "test_end": f["test_end"],
        "train_bars": len(f["train_df"]),
        "test_bars": len(f["test_df"]),
    }
    for i, f in enumerate(folds)
])

print("folds:", len(folds))
folds_info


## 10. Walk-forward optimization

In [ ]:
def run_walk_forward(
    folds: List[Dict[str, Any]],
    base: StrategyParams = BASELINE_PARAMS,
    n_trials_per_fold: int = 150,
    start_cash: float = 10_000.0,
    commission: float = 0.0004,
    broker_leverage: float = 100.0,
    sampler_seed: int = 42,
    min_trades: int = 50,
    max_dd_pct: float = 30.0,
) -> pd.DataFrame:
    if optuna is None:
        raise ImportError("Optuna не установлена: pip install optuna")

    rows = []

    for fold_id, fold in enumerate(folds):
        print(
            f"Fold {fold_id}: "
            f"train {fold['train_start']} → {fold['train_end']} | "
            f"test {fold['test_start']} → {fold['test_end']}"
        )

        sampler = optuna.samplers.TPESampler(seed=sampler_seed + fold_id)
        study = optuna.create_study(direction="maximize", sampler=sampler)

        objective = make_objective(
            fold["train_df"],
            base=base,
            start_cash=start_cash,
            commission=commission,
            broker_leverage=broker_leverage,
            min_trades=min_trades,
            max_dd_pct=max_dd_pct,
        )

        study.optimize(
            objective,
            n_trials=n_trials_per_fold,
            show_progress_bar=False,
            gc_after_trial=True,
        )

        best_params = params_from_trial_params(
            study.best_trial.params,
            base=base,
        )

        train_metrics, _, _ = run_backtest(
            fold["train_df"],
            best_params,
            start_cash=start_cash,
            commission=commission,
            broker_leverage=broker_leverage,
        )

        test_metrics, _, _ = run_backtest(
            fold["test_df"],
            best_params,
            start_cash=start_cash,
            commission=commission,
            broker_leverage=broker_leverage,
        )

        row = {
            "fold": fold_id,
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            "study_best_value": study.best_value,
            "study_best_trial": study.best_trial.number,
            **{f"param_{k}": v for k, v in asdict(best_params).items()},
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"test_{k}": v for k, v in test_metrics.items()},
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
wfa_smoke = run_walk_forward(
    folds,
    base=WFA_BASE_PARAMS,
    n_trials_per_fold=300,
    start_cash=10_000,
    commission=0.0004,
    broker_leverage=100,
    sampler_seed=42,
    min_trades=25,
    max_dd_pct=40,
)

wfa_smoke.head()


In [ ]:
pd.Series({
    "folds": len(wfa_smoke),
    "positive_test_folds": (wfa_smoke["test_net_pnl_pct"] > 0).sum(),
    "positive_test_folds_pct": (wfa_smoke["test_net_pnl_pct"] > 0).mean() * 100,
    "avg_test_pnl_pct": wfa_smoke["test_net_pnl_pct"].mean(),
    "median_test_pnl_pct": wfa_smoke["test_net_pnl_pct"].median(),
    "avg_test_pf": wfa_smoke["test_profit_factor"].replace([np.inf, -np.inf], np.nan).mean(),
    "median_test_pf": wfa_smoke["test_profit_factor"].replace([np.inf, -np.inf], np.nan).median(),
    "max_test_dd_pct": wfa_smoke["test_max_drawdown_pct"].max(),
    "total_test_trades": wfa_smoke["test_total_trades"].sum(),
})

In [ ]:
wfa_smoke[[
    "fold",
    "test_start",
    "test_end",
    "param_rsi_len",
    "param_left_bars",
    "param_right_bars",
    "param_rr",
    "param_stop_buffer_pct",
    "param_max_setup_bars",
    "train_net_pnl_pct",
    "train_profit_factor",
    "train_total_trades",
    "test_net_pnl_pct",
    "test_profit_factor",
    "test_max_drawdown_pct",
    "test_total_trades",
]].sort_values("test_net_pnl_pct")

## 11. WFA vs fixed baseline

In [ ]:
baseline_rows = []

for i, fold in enumerate(folds):
    test_metrics, _, _ = run_backtest(
        fold["test_df"],
        BASELINE_PARAMS,
        start_cash=10_000,
        commission=0.0004,
        broker_leverage=100,
    )

    baseline_rows.append({
        "fold": i,
        "test_start": fold["test_start"],
        "test_end": fold["test_end"],
        **{f"baseline_{k}": v for k, v in test_metrics.items()},
    })

baseline_wfa_compare = pd.DataFrame(baseline_rows)

compare = wfa_smoke.merge(
    baseline_wfa_compare,
    on=["fold", "test_start", "test_end"],
    how="left"
)

compare[[
    "fold",
    "test_start",
    "test_end",
    "test_net_pnl_pct",
    "baseline_net_pnl_pct",
    "test_profit_factor",
    "baseline_profit_factor",
    "test_max_drawdown_pct",
    "baseline_max_drawdown_pct",
    "test_total_trades",
    "baseline_total_trades",
]]

In [ ]:
pd.Series({
    "wfa_positive_folds_pct": (compare["test_net_pnl_pct"] > 0).mean() * 100,
    "baseline_positive_folds_pct": (compare["baseline_net_pnl_pct"] > 0).mean() * 100,

    "wfa_median_pnl": compare["test_net_pnl_pct"].median(),
    "baseline_median_pnl": compare["baseline_net_pnl_pct"].median(),

    "wfa_median_pf": compare["test_profit_factor"].replace([np.inf, -np.inf], np.nan).median(),
    "baseline_median_pf": compare["baseline_profit_factor"].replace([np.inf, -np.inf], np.nan).median(),

    "wfa_max_dd": compare["test_max_drawdown_pct"].max(),
    "baseline_max_dd": compare["baseline_max_drawdown_pct"].max(),
})

## 12. Rolling validation candidates

In [ ]:
from dataclasses import replace

candidate_params = {
    "baseline_13_3_5_15_buf0_mb100": replace(
        BASELINE_PARAMS,
        rsi_len=13,
        left_bars=3,
        right_bars=5,
        rr=1.5,
        stop_buffer_pct=0.0,
        max_setup_bars=100,
    ),
    "baseline_13_3_5_15_buf0_mb90": replace(
        BASELINE_PARAMS,
        rsi_len=13,
        left_bars=3,
        right_bars=5,
        rr=1.5,
        stop_buffer_pct=0.0,
        max_setup_bars=90,
    ),
    "conservative_11_4_5_15_buf005_mb90": replace(
        BASELINE_PARAMS,
        rsi_len=11,
        left_bars=4,
        right_bars=5,
        rr=1.5,
        stop_buffer_pct=0.05,
        max_setup_bars=90,
    ),
    "conservative_12_4_5_15_buf005_mb90": replace(
        BASELINE_PARAMS,
        rsi_len=12,
        left_bars=4,
        right_bars=5,
        rr=1.5,
        stop_buffer_pct=0.05,
        max_setup_bars=90,
    ),
    "slower_13_4_5_15_buf0_mb100": replace(
        BASELINE_PARAMS,
        rsi_len=13,
        left_bars=4,
        right_bars=5,
        rr=1.5,
        stop_buffer_pct=0.0,
        max_setup_bars=100,
    ),
}

candidate_rows = []

for name, params in candidate_params.items():
    for i, fold in enumerate(folds):
        metrics, _, _ = run_backtest(
            fold["test_df"],
            params,
            start_cash=10_000,
            commission=0.0004,
            broker_leverage=100,
        )

        candidate_rows.append({
            "candidate": name,
            "fold": i,
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            **metrics,
        })

candidate_results = pd.DataFrame(candidate_rows)
candidate_results.head()

In [ ]:
candidate_summary = candidate_results.groupby("candidate").agg(
    folds=("fold", "count"),
    positive_folds=("net_pnl_pct", lambda x: (x > 0).sum()),
    positive_folds_pct=("net_pnl_pct", lambda x: (x > 0).mean() * 100),
    avg_pnl_pct=("net_pnl_pct", "mean"),
    median_pnl_pct=("net_pnl_pct", "median"),
    avg_pf=("profit_factor", lambda x: x.replace([np.inf, -np.inf], np.nan).mean()),
    median_pf=("profit_factor", lambda x: x.replace([np.inf, -np.inf], np.nan).median()),
    max_dd_pct=("max_drawdown_pct", "max"),
    avg_dd_pct=("max_drawdown_pct", "mean"),
    total_trades=("total_trades", "sum"),
)

candidate_summary.sort_values(
    ["positive_folds_pct", "median_pf", "median_pnl_pct"],
    ascending=False
)

## 13. Risk sweep

In [ ]:
ROBUST_PARAMS = replace(
    BASELINE_PARAMS,
    rsi_len=13,
    left_bars=3,
    right_bars=5,
    rr=1.5,
    stop_buffer_pct=0.0,
    max_setup_bars=90,
)

risk_values = [0.5, 1.0, 1.5, 2.0, 3.0]

risk_rows = []

for risk in risk_values:
    params = replace(ROBUST_PARAMS, risk_pct=risk)

    for i, fold in enumerate(folds):
        metrics, _, _ = run_backtest(
            fold["test_df"],
            params,
            start_cash=10_000,
            commission=0.0004,
            broker_leverage=100,
        )

        risk_rows.append({
            "risk_pct": risk,
            "fold": i,
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            **metrics,
        })

risk_results = pd.DataFrame(risk_rows)

risk_summary = risk_results.groupby("risk_pct").agg(
    folds=("fold", "count"),
    positive_folds=("net_pnl_pct", lambda x: (x > 0).sum()),
    positive_folds_pct=("net_pnl_pct", lambda x: (x > 0).mean() * 100),
    avg_pnl_pct=("net_pnl_pct", "mean"),
    median_pnl_pct=("net_pnl_pct", "median"),
    avg_pf=("profit_factor", lambda x: x.replace([np.inf, -np.inf], np.nan).mean()),
    median_pf=("profit_factor", lambda x: x.replace([np.inf, -np.inf], np.nan).median()),
    max_dd_pct=("max_drawdown_pct", "max"),
    avg_dd_pct=("max_drawdown_pct", "mean"),
    total_trades=("total_trades", "sum"),
)

risk_summary

In [ ]:
full_risk_rows = []

for risk in [0.5, 1.0, 1.5, 2.0, 3.0]:
    params = replace(
        ROBUST_PARAMS,
        risk_pct=risk,
    )

    metrics, strat, cerebro = run_backtest(
        df,
        params,
        start_cash=10_000,
        commission=0.0004,
        broker_leverage=100,
    )

    full_risk_rows.append({
        "risk_pct": risk,
        **metrics,
    })

full_risk_summary = pd.DataFrame(full_risk_rows)

full_risk_summary[[
    "risk_pct",
    "net_pnl_pct",
    "max_drawdown_pct",
    "total_trades",
    "winrate_pct",
    "profit_factor",
    "long_profit_factor",
    "short_profit_factor",
]]

## 14. Commission stress-test

In [ ]:
stress_rows = []

for commission in [0.0004, 0.0006, 0.0008, 0.0010]:
    for risk in [1.0, 2.0]:
        params = replace(
            ROBUST_PARAMS,
            risk_pct=risk,
        )

        metrics, strat, cerebro = run_backtest(
            df,
            params,
            start_cash=10_000,
            commission=commission,
            broker_leverage=100,
        )

        stress_rows.append({
            "risk_pct": risk,
            "commission": commission,
            **metrics,
        })

stress_summary = pd.DataFrame(stress_rows)

stress_summary[[
    "risk_pct",
    "commission",
    "net_pnl_pct",
    "max_drawdown_pct",
    "total_trades",
    "winrate_pct",
    "profit_factor",
    "long_profit_factor",
    "short_profit_factor",
]]